# cudf-bench: allocator transient probe (Step 3c)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. ~8 min. **Click Allow on the download at the end.**

Finding under test: skewed cuDF groupby is slow for the first few calls in a process, then settles (see results/probe.csv rep times). Hypothesis: the GPU **memory pool growing on demand** — raw GPU allocation is expensive, and skewed aggregation seems to demand sizes the pool doesn't have yet.

Test: time 12 consecutive groupby calls **including the cold first call**, with the default allocator vs a pool pre-grown to 8 GiB before any work.

Predictions if the allocator is guilty: default-pool skewed curve starts high and decays; prealloc curves flat from call 1; uniform-key curves flat either way.

In [ ]:
!nvidia-smi -L

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
# VM clone is disposable — force it to match GitHub exactly, discard any leftovers
!git fetch -q && git reset -q --hard origin/main && git clean -qfd results
!rm -f results/transient.csv
%pip install -q -e .

## Runs (each line = fresh process, 12 timed calls incl. cold start)

In [ ]:
# baseline shape reference: no strings, default allocator
for sk in ["0", "1.5"]:
    !python -m benchmark.transient --skew {sk} --str-cols 0 --rmm-pool default --iters 12 --out results/transient.csv
# the 2x2 that answers the question: strings present, allocator default vs pre-grown
for pool in ["default", "prealloc"]:
    for sk in ["0", "1.5"]:
        !python -m benchmark.transient --skew {sk} --str-cols 2 --rmm-pool {pool} --iters 12 --out results/transient.csv

## Quick look: per-call curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("results/transient.csv")
fig, ax = plt.subplots(figsize=(9, 4.5))
for (skew, sc, pool), g in df.groupby(["skew", "str_cols", "rmm_pool"]):
    ax.plot(g["iter"], g["time_s"], marker="o", label=f"skew={skew} str={sc} {pool}")
ax.set_xlabel("call #"); ax.set_ylabel("seconds"); ax.set_title("12 consecutive groupby calls (10M rows)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Auto-download (click Allow)

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/transient.csv")